# The effect of speech rate on coherence

Does EEG-envelope coherence sit where the speech does? Figure 2's grand averages pool
every stimulus together; this notebook cuts the data by how fast the speech is and asks
whether the coherence spectrum moves with it.

Ported from `exploration/3_speech_units_v_coh.ipynb`, which carries the exploratory
work this selects from. Kept here: the per-stimulus reduction, the equal-duration
grouping, the band fits, and the three figures below. Left there: the per-stimulus
correlation tables, the rate pair-plot, the estimator calibration, and the supplementary
figures that partial listening time or the null floor out of the last one.

## Figures produced

1. **`speech_rate_and_coherence_<reference>`** -- the main figure. (a) the
   stimulus-level coherence heatmap ordered by syllable rate, with each unit's rate
   drawn on it; (b) the grouped spectra for one exemplar unit; (c)-(e) the fitted peak
   frequency against each unit rate, in the 0.2-2 Hz band, again with intonation-unit
   rate partialled out, and in the 4-15 Hz band.
2. **`grouped_spectra_by_unit_<reference>`** -- panel (b) of the above, for all four
   units, with each group's rate and listening time captioned.
3. **`band_rms_vs_rate_<reference>`** -- how *strong* each band is against the unit
   rates, rather than where it peaks.
4. **`band_peak_height_vs_rate_<reference>`** -- the same, with strength measured as the
   coherence at the band's fitted peak rather than its average over the band.

## Setup

`REFERENCE` selects the EEG reference. Every cache and every figure name follows from
it, so one reference can never read another's cache or overwrite its figures. The
inputs are the per-stimulus null files from `get_null_gammatone_coherence.py`
(`null_coherence_dir/gammatone_null_by_stimulus`) and the per-stimulus unit rates from
`scripts/speech_units_pitch/`.

The house style is figure 2's, so the two carry text at the same physical size.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from scipy.stats import false_discovery_control, pearsonr, t as t_dist
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "figures" else Path.cwd()
CONFIG = json.loads((ROOT / "src" / "package_data" / "config.json").read_text())

REFERENCE = "avg"  # "avg" | "linked_mastoids" | "FCz"

sys.path.insert(0, str(ROOT / "src"))
from coherence.pipeline import resolve_reference_dirs

OUT_DIR = ROOT / "figures"
# The same cache tree figure 2 uses, one directory per reference.
CACHE_ROOT = OUT_DIR / "figure_data"


def cache_dir(reference):
    path = CACHE_ROOT / reference
    path.mkdir(parents=True, exist_ok=True)
    return path


STIMULUS_NULL_DIR = (resolve_reference_dirs(CONFIG, REFERENCE, ROOT)[1]
                     / "gammatone_null_by_stimulus")
SPEECH_UNITS_DIR = Path(CONFIG["speech_units_dir"])

# --- house style, shared with figures 1 and 2 --------------------------------
DEFAULT_FONTSIZE = 10
TICK_FONTSIZE = 8
SMALL_FONTSIZE = 6.5
PANEL_LABEL_FONTSIZE = 14

INK, MUTED, HAIRLINE = "#1b1b1a", "#6b6b68", "#d5d5d1"
NULL_FILL = "#b9b9b4"
MEDIAN_COLOUR = "#c1272d"

plt.rcParams.update({
    "font.size": TICK_FONTSIZE,
    "axes.labelsize": DEFAULT_FONTSIZE,
    "axes.titlesize": DEFAULT_FONTSIZE,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "axes.linewidth": 0.8,
    "figure.dpi": 130,
    # TrueType, not matplotlib's default Type 3, which publishers flag. Has to be set
    # before anything is drawn.
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def bare(ax):
    """Recessive chrome: drop the top/right rules."""
    ax.spines[["top", "right"]].set_visible(False)
    return ax


print(f"reference: {REFERENCE}  "
      f"({len(sorted(STIMULUS_NULL_DIR.glob('*.npz')))} stimulus null files)")

## Referencing each stimulus to its own null

Per stimulus, in three steps -- identical to the participant-level construction in
figure 2 of notebook 2, one axis over:

1. the stimulus's duration-weighted **complex-average** coherence, reduced to an **RMS
   magnitude** over the 64 sensors;
2. minus that stimulus's own **median null RMS, by frequency**, over its 500 draws;
3. divided by the **RMS of that centred trace**, one scalar per stimulus.

Step 3 puts each stimulus on its own scale, so the heatmap rows carry the *shape* of a
stimulus's coherence spectrum rather than its size. The null draws take the identical
treatment, so the null stays on the stimulus's scale.

The **un-standardised** RMS trace and the null median are kept alongside, because the
band measures further down are defined on the coherence as measured -- the divisor is
deliberately absent there.

In [ ]:
BAND = (0.0, 30.0)          # the heatmap's frequency range


def pool_stimuli(paths, analysed):
    """Each stimulus's RMS trace, referenced to and scaled by its own null.

    Returns the standardised traces ``(n_stimuli, F)``; the un-standardised RMS and the
    null median, both ``(n_stimuli, F)``; each stimulus's scale, duration and name.
    """
    norm, rms, null_medians, scales, seconds, names = [], [], [], [], [], []

    for path in tqdm(paths, desc="stimuli", unit="file"):
        with np.load(path) as saved:
            # (1) complex-average coherence -> RMS magnitude over the sensors.
            real = saved["gammatone_coherence"].squeeze(1)                # (F, n_eeg)
            real_rms = np.sqrt((np.abs(real) ** 2).mean(-1))              # (F,)
            seconds.append(float(saved["total_analysed_duration"]))
            names.append(str(saved["name"]))

            # In place: a second (D, F, n_eeg) temporary is ~0.3 GB per stimulus.
            power = np.abs(saved["null_gammatone_coherence"].squeeze(2))
            power **= 2
            null_rms = np.sqrt(power.mean(-1))                            # (D, F)
        del power

        # (2) remove this stimulus's by-frequency null median, then (3) divide by the
        # RMS of what is left.
        null_median = np.median(null_rms, axis=0)
        centred = real_rms - null_median
        scale = float(np.sqrt((centred[analysed] ** 2).mean()))

        norm.append((centred / scale).astype(np.float32))
        rms.append(real_rms.astype(np.float32))
        null_medians.append(null_median.astype(np.float32))
        scales.append(scale)

    return (np.array(norm), np.array(rms), np.array(null_medians),
            np.array(scales), np.array(seconds), names)


def stimulus_level(reference):
    """The cached per-stimulus reduction, building it on first use.

    The cache carries the stimuli it was built from, so a directory that later gains or
    loses one is re-pooled rather than read back with the old rows.
    """
    paths = sorted((resolve_reference_dirs(CONFIG, reference, ROOT)[1]
                    / "gammatone_null_by_stimulus").glob("*.npz"))
    expected = [path.stem for path in paths]
    cache = cache_dir(reference) / "stimulus_coherence.npz"

    if cache.exists():
        with np.load(cache) as saved:
            if [str(name) for name in saved["files"]] == expected:
                return {key: saved[key] for key in
                        ("frequencies", "norm", "rms", "null_median", "scale",
                         "seconds")} | dict(names=[str(n) for n in saved["names"]])
        print(f"  {reference}: cached stimuli differ, re-pooling")

    with np.load(paths[0]) as first:
        axis = first["frequencies"]
    norm, rms, null_median, scale, seconds, names = pool_stimuli(paths, axis > 0)
    np.savez(cache, frequencies=axis, norm=norm, rms=rms, null_median=null_median,
             scale=scale, seconds=seconds, names=np.array(names),
             files=np.array(expected))
    return dict(frequencies=axis, norm=norm, rms=rms, null_median=null_median,
                scale=scale, seconds=seconds, names=names)


stimuli = stimulus_level(REFERENCE)
frequencies = stimuli["frequencies"]
stimulus_norm = stimuli["norm"]                 # (n_stimuli, F), standardised
stimulus_rms = stimuli["rms"]                   # (n_stimuli, F), as measured
stimulus_null_median = stimuli["null_median"]
stimulus_seconds = stimuli["seconds"]
stimulus_names = stimuli["names"]

in_band = (frequencies > BAND[0]) & (frequencies <= BAND[1])
analysed = frequencies > 0                      # DC is dropped, as everywhere else

print(f"{len(stimulus_names)} stimuli, {stimulus_seconds.sum() / 3600:.1f} analysed "
      f"hours; per stimulus {stimulus_seconds.min() / 60:.1f}-"
      f"{stimulus_seconds.max() / 60:.1f} min")

In [ ]:
# The pipeline's own per-stimulus rates -- n_units / duration, written by
# scripts/speech_units_pitch/ into the directory figure 2's unit-rate panel reads.
# Nothing is recomputed here.
#
# These are whole-stimulus mean rates, pauses included. That is a different summary
# from figure 2's panel (b), which histograms *windowed* instantaneous rates and quotes
# the modal one; the modal rate is the higher of the two, because it ignores the pauses
# that drag a whole-stimulus mean down.
UNIT_RATE_FILES = {
    "Int. units": "iu_rates.json",
    "Words": "word_rates.json",
    "Syllables": "syllabic_rates_heuristic.json",
    "Phonemes": "phoneme_rates.json",
}
UNIT_SLUGS = {"Int. units": "iu", "Words": "word",
              "Syllables": "syllable", "Phonemes": "phoneme"}
# Read as "Group mean <name> rate", so a legend names the unit rather than saying
# "unit rate" and leaving the reader to work out which panel they are in.
UNIT_LONG_NAMES = {"Int. units": "intonation unit", "Words": "word",
                   "Syllables": "syllable", "Phonemes": "phoneme"}
# Figure 2's unit colours, so a unit is the same colour in both notebooks.
UNIT_COLOURS = ["#ef476f", "#ffd166", "#06d6a0", "#118ab2"]

unit_rates = {}
for label, filename in UNIT_RATE_FILES.items():
    by_stimulus = json.loads((SPEECH_UNITS_DIR / filename).read_text())
    unit_rates[label] = np.array([by_stimulus[name] for name in stimulus_names])

rate_by_unit = {UNIT_SLUGS[label]: rates for label, rates in unit_rates.items()}

for label, rates in unit_rates.items():
    print(f"{label:11s} {rates.min():5.2f}-{rates.max():5.2f} Hz "
          f"(median {np.median(rates):5.2f})")

## Grouping stimuli

A single stimulus does not carry enough data to locate a spectral peak: the estimator
noise on a per-stimulus peak was measured at ~3 Hz against a 1.4 Hz spread across
stimuli. Pooling buys the signal back. Stimuli are sorted by a unit rate and cut into
exactly `N_GROUPS` blocks of as near to equal listening time as the durations allow,
and the analysis runs on the groups.

Two things matter about how it is done:

* **The pooling is on the complex coherence**, duration-weighted, before any magnitude
  is taken -- the same order the grand averages use. That lets noise cancel rather than
  accumulate, which is what makes a group's spectrum cleaner than its members'. The
  null floor falls from about half the observed value per stimulus to under a fifth per
  group in 0.2-2 Hz.
* **The groups are rebuilt for every unit.** Sorting is by that unit's rate, so the
  membership under `syllable` is not the membership under `iu`, and the four analyses
  are not four views of one partition.

The cost is sample size: `N_GROUPS` points per unit, so `|r|` has to reach about 0.7 to
clear p < 0.05 at n = 8.

In [ ]:
# Every unit is split into the SAME number of groups, so the four correlations rest on
# the same n and the panels are directly comparable. Filling groups to a minimum
# listening time instead left each unit with whatever count fell out and dumped the
# remainder into an oversized last group.
N_GROUPS = 8

# Order of the polynomial fitted across each band to locate its peak.
POLYNOMIAL_ORDER = 4

GROUP_BANDS = {"delta": (0.2, 2.0), "theta": (4.0, 15.0)}
SPECTRUM_RANGE = (0.2, 30.0)     # contiguous, so the pooled spectra can be drawn


def form_groups(rate, seconds, n_groups):
    """Stimuli sorted by ascending `rate`, cut into exactly `n_groups` contiguous
    blocks carrying as near to equal listening time as the durations allow.

    Exact, by dynamic programming: ``cost[i, k]`` is the least achievable sum of
    squared deviations from the per-group target once the first `i` stimuli fill `k`
    blocks. A greedy accumulate-to-a-threshold walk cannot match this -- it commits to
    early boundaries it can never revisit, which is what left the old rule with 15 h
    groups at one end and a 33 h group at the other.

    One stimulus (the one every participant heard) runs to 18.7 h on its own, past the
    target, so it is always handed a block of its own. That is left standing: it
    carries more listening than most groups, it simply pools across participants rather
    than across stimuli.
    """
    order = np.argsort(rate)
    weights = seconds[order]
    n = len(weights)
    prefix = np.concatenate([[0.0], np.cumsum(weights)])
    target = prefix[-1] / n_groups

    cost = np.full((n + 1, n_groups + 1), np.inf)
    split = np.zeros((n + 1, n_groups + 1), dtype=int)
    cost[0, 0] = 0.0
    for k in range(1, n_groups + 1):
        # Leave enough stimuli behind to fill the blocks still to come.
        for i in range(k, n - (n_groups - k) + 1):
            starts = np.arange(k - 1, i)
            candidate = cost[starts, k - 1] + (prefix[i] - prefix[starts] - target) ** 2
            best = int(np.argmin(candidate))
            cost[i, k] = candidate[best]
            split[i, k] = starts[best]

    groups, i = [], n
    for k in range(n_groups, 0, -1):
        j = split[i, k]
        groups.append(list(order[j:i]))
        i = j
    return groups[::-1]


groupings = {slug: form_groups(rates, stimulus_seconds, N_GROUPS)
             for slug, rates in rate_by_unit.items()}

# Only a narrow range is needed, so the frequency axis is sliced on load: that turns
# each group's null accumulator from 655 MB into 38 MB, which is what lets all four
# groupings be pooled in one pass over the files rather than four.
union_mask = (frequencies >= SPECTRUM_RANGE[0]) & (frequencies <= SPECTRUM_RANGE[1])
union_frequencies = frequencies[union_mask]
band_within_union = {name: (union_frequencies >= low) & (union_frequencies <= high)
                     for name, (low, high) in GROUP_BANDS.items()}


def pool_groups(names, paths, groupings, mask):
    """Each group's duration-weighted complex-average coherence, as an RMS magnitude
    over the sensors, and the median of its null put through the same order.

    One pass over the stimulus files: a stimulus is added to every group that claims
    it, across all four groupings at once.
    """
    keys = [(slug, i) for slug, groups in groupings.items() for i in range(len(groups))]
    accumulators = {key: None for key in keys}
    claims = {index: [(slug, i) for slug, groups in groupings.items()
                      for i, members in enumerate(groups) if index in members]
              for index in range(len(names))}

    for index, name in enumerate(tqdm(names, desc="stimuli", unit="file")):
        with np.load(paths[name]) as saved:
            weight = float(saved["total_analysed_duration"])
            real = saved["gammatone_coherence"].squeeze(1)[mask]               # (Fb, ch)
            null = saved["null_gammatone_coherence"].squeeze(2)[:, mask]   # (D, Fb, ch)
        for key in claims[index]:
            if accumulators[key] is None:
                accumulators[key] = [np.zeros(real.shape, np.complex128),
                                     np.zeros(null.shape, np.complex64), 0.0]
            accumulators[key][0] += real * weight
            accumulators[key][1] += null * np.float32(weight)
            accumulators[key][2] += weight
        del real, null

    rows, real_rms_all, null_median_all = [], [], []
    for slug, groups in groupings.items():
        for i, members in enumerate(groups):
            real_sum, null_sum, weight_sum = accumulators[(slug, i)]
            real_rms = np.sqrt((np.abs(real_sum / weight_sum) ** 2).mean(-1))
            power = np.abs(null_sum / np.complex64(weight_sum))
            power **= 2
            null_rms = np.sqrt(power.mean(-1))                                 # (D, Fb)
            del power
            real_rms_all.append(real_rms)
            null_median_all.append(np.median(null_rms, axis=0))
            rows.append({"unit": slug, "group": i, "n_stimuli": len(members),
                         "hours": weight_sum / 3600,
                         "mean_rate": float(np.average(
                             rate_by_unit[slug][members],
                             weights=stimulus_seconds[members])),
                         "members": "|".join(names[j] for j in members)})
            accumulators[(slug, i)] = None      # freed as we go: 32 x 38 MB otherwise

    return pd.DataFrame(rows), np.array(real_rms_all), np.array(null_median_all)


def grouped_spectra(reference):
    """The cached grouping for one reference, building it on first use.

    Keyed on the group count and the frequency range as well as the stimuli: both
    change the shape or the membership of what is stored, and a cache built under
    other values would load with the right shape and the wrong content.
    """
    cache = (cache_dir(reference) /
             f"grouped_spectra_{N_GROUPS}groups"
             f"_{SPECTRUM_RANGE[0]:g}-{SPECTRUM_RANGE[1]:g}Hz.npz")
    if cache.exists():
        with np.load(cache, allow_pickle=True) as saved:
            if [str(n) for n in saved["stimuli"]] == list(stimulus_names):
                table = pd.DataFrame(saved["table"],
                                     columns=[str(c) for c in saved["columns"]])
                table = table.astype({c: float for c in table.columns
                                      if c not in ("unit", "members")})
                return table, saved["real_rms"], saved["null_median"]
        print(f"  {reference}: cached stimuli differ, re-pooling")

    directory = (resolve_reference_dirs(CONFIG, reference, ROOT)[1]
                 / "gammatone_null_by_stimulus")
    paths = {name: directory / f"{name}_gammatone_null.npz" for name in stimulus_names}
    table, real_rms, null_median = pool_groups(stimulus_names, paths, groupings,
                                               union_mask)
    np.savez(cache, table=table.to_numpy().astype(object),
             columns=np.array(table.columns, dtype=object),
             real_rms=real_rms, null_median=null_median,
             spectrum_frequencies=union_frequencies,
             stimuli=np.array(stimulus_names))
    return table, real_rms, null_median


group_table, group_real_rms, group_null_median = grouped_spectra(REFERENCE)
group_excess = group_real_rms - group_null_median

print(f"{N_GROUPS} groups per unit, equal-duration partition; listening hours per group:")
for slug in UNIT_SLUGS.values():
    hours = group_table.loc[group_table.unit == slug, "hours"]
    print(f"  {slug:9s} {hours.min():5.1f}-{hours.max():5.1f} h (sd {hours.std():4.1f}), "
          f"stimuli per group "
          + str(list(group_table.loc[group_table.unit == slug, "n_stimuli"].astype(int))))

In [ ]:
# --- locating a band's peak ------------------------------------------------------


def band_polynomial_fit(band_frequencies, values, degree=None):
    """Least-squares polynomial across a whole band. Returns (coefficients, span).

    The band is mapped onto [-1, 1] before fitting. That is not cosmetic: a polynomial
    in raw Hz over 4-15 is badly conditioned, increasingly so as the order rises.
    """
    degree = POLYNOMIAL_ORDER if degree is None else degree
    # Never let the fit approach interpolation: with 10 bins in the delta band, an
    # order-8 fit has 9 free parameters and reaches R2 = 1.00 by construction.
    degree = min(degree, len(band_frequencies) - 2)
    low, high = band_frequencies[0], band_frequencies[-1]
    scaled = 2 * (band_frequencies - low) / (high - low) - 1
    return np.polyfit(scaled, values, degree), (low, high)


def evaluate_band_fit(coefficients, span, frequencies_out):
    """The fitted curve at arbitrary frequencies inside the band."""
    low, high = span
    return np.polyval(coefficients, 2 * (frequencies_out - low) / (high - low) - 1)


def band_polynomial_peak(band_frequencies, values, degree=None):
    """Maximum of a polynomial fitted across the whole band, with the fit's R2.

    Uses every bin rather than a window around the argmax, so there is no window
    parameter and no dependence on which single bin came out highest. Where the fit has
    several interior maxima -- possible once the order exceeds 3 -- the highest is taken.
    """
    coefficients, span = band_polynomial_fit(band_frequencies, values, degree)
    fitted = evaluate_band_fit(coefficients, span, band_frequencies)
    residual = 1 - (((values - fitted) ** 2).sum()
                    / ((values - values.mean()) ** 2).sum())

    roots = np.roots(np.polyder(coefficients))
    roots = roots[np.abs(roots.imag) < 1e-9].real
    curvature = np.polyval(np.polyder(np.polyder(coefficients)), roots)
    inside = roots[(curvature < 0) & (roots >= -1) & (roots <= 1)]
    if inside.size == 0:                    # no interior maximum: the fit's best bin
        return band_frequencies[np.argmax(fitted)], residual
    low, high = span
    best = inside[np.argmax(np.polyval(coefficients, inside))]
    return low + (best + 1) * (high - low) / 2, residual


# --- the band measures the figures correlate -------------------------------------
# What the fits are computed on -- ONE LINE, and it moves the peaks, the correlations
# and the drawn fits together:
#   True : the coherence as measured, which is what the stacked panels draw, so a fit
#          and the trace it is drawn over are the same quantity.
#   False: the null-corrected excess. The floor is a third to a half of the measured
#          coherence here and is not a constant offset -- where it slopes across a band
#          it tilts the fit and moves its maximum.
FIT_ON_MEASURED = True

# Row scaling for the stacked-spectra panels, set here so the main figure and the
# supplementary one cannot disagree about it.
#   True : every row divided by the SAME number, so a row's height is its coherence and
#          rows are comparable with each other and across panels. The cost is that the
#          group with the strongest delta peak sets the scale.
#   False: each row divided by its own maximum. Every row fills its band, but the
#          heights mean nothing -- a tall row is not a more coherent group.
COMMON_ROW_SCALE = True
GROUP_ROW_SCALE = np.abs(group_real_rms).max()

band_source = group_real_rms if FIT_ON_MEASURED else group_excess

for band, within in band_within_union.items():
    band_f = union_frequencies[within]
    values = band_source[:, within]
    fitted = [band_polynomial_peak(band_f, row) for row in values]
    group_table[f"peak_{band}_bandfit"] = [peak for peak, _ in fitted]
    group_table[f"r2_{band}"] = [r2 for _, r2 in fitted]
    # The band's coherence magnitude, averaged over its bins. `peak_*` says where in
    # the band the response sits; this says how strong it is, which is the other
    # question the third figure asks.
    group_table[f"rms_{band}"] = values.mean(1)
    # The same on the null-corrected excess. Unlike a peak frequency, a magnitude is
    # displaced directly by the floor under it, and the floor tracks listening time,
    # which is not uniform across groups.
    group_table[f"rms_{band}_excess"] = group_excess[:, within].mean(1)
    group_table[f"floor_{band}"] = (group_null_median[:, within].mean(1)
                                    / group_real_rms[:, within].mean(1))

# Each group's mean rate for EVERY unit, not just its own -- the covariate the partial
# row of the main figure needs.
stimulus_index = {name: i for i, name in enumerate(stimulus_names)}
group_members = [[stimulus_index[name] for name in members.split("|")]
                 for members in group_table.members]
for slug in UNIT_SLUGS.values():
    group_table[f"mean_rate_{slug}"] = [
        float(np.average(rate_by_unit[slug][members], weights=stimulus_seconds[members]))
        for members in group_members]

print(f"band measures computed on the "
      f"{'measured spectrum' if FIT_ON_MEASURED else 'null-corrected excess'}")
for band, (low, high) in GROUP_BANDS.items():
    n_bins = int(band_within_union[band].sum())
    print(f"  {band:6s} ({low}-{high} Hz, {n_bins} bins): null floor "
          f"{group_table[f'floor_{band}'].mean() * 100:4.1f}% of the observed RMS; "
          f"order-{POLYNOMIAL_ORDER} fit R2 {group_table[f'r2_{band}'].mean():.2f}, "
          f"peak {group_table[f'peak_{band}_bandfit'].mean():.2f} Hz "
          f"(sd {group_table[f'peak_{band}_bandfit'].std():.2f})")

In [ ]:
def partial_correlation(x, y, z):
    """Pearson correlation of x and y with z partialled out of both."""
    r_xy, r_xz, r_yz = (pearsonr(x, y).statistic, pearsonr(x, z).statistic,
                        pearsonr(y, z).statistic)
    r = (r_xy - r_xz * r_yz) / np.sqrt((1 - r_xz ** 2) * (1 - r_yz ** 2))
    n = len(x)
    statistic = r * np.sqrt((n - 3) / (1 - r ** 2))
    return r, 2 * t_dist.sf(abs(statistic), n - 3)


def control_residuals(values, covariate):
    """`values` with `covariate` regressed out, by least squares.

    Plotting these against each other is the partial correlation's counterpart of a
    plain scatter: the Pearson r of the two residual sets is exactly the partial
    Pearson r reported beside it, which the figures assert rather than assume.
    """
    slope, intercept = np.polyfit(covariate, values, 1)
    return values - (slope * covariate + intercept)


def correct(pvalues):
    """Benjamini-Hochberg FDR WITHIN a row of panels.

    Each row is its own family: one measure against each of the unit rates, so the
    family is 4, or 3 in the partial row where IU cannot be partialled out of itself.
    BH rather than Bonferroni because the four rates correlate with each other up to
    r = 0.99 -- these are nowhere near independent tests, and a factor-of-n correction
    would be far too harsh.
    """
    return list(false_discovery_control(np.asarray(pvalues), method="bh"))

## Figure 1 -- speech rate and coherence

Full text-block width on A4 with 1 in margins (6.27 in), the same width figures 1 and 2
use, so all three carry text at the same physical size. The height leaves roughly two
inches of the text block for a caption.

* **(a)** every stimulus's standardised coherence, rows ordered by syllable rate, with
  each unit's rate drawn on the frequency axis as a marker per stimulus.
* **(b)** the grouped spectra for one exemplar unit, each row a group, with the band
  fits and their peaks marked and the group's mean rate as a coloured rule.
* **(c)** the fitted 0.2-2 Hz peak against each unit rate, one panel per unit.
* **(d)** the fitted 4-15 Hz peak against each unit rate.

The partial row the exploratory notebook draws -- (c) with intonation-unit rate
partialled out -- is **computed and printed but not drawn**. It is worth keeping in the
output: the delta effect survives partialling IU out for words and phonemes, which is
the answer to the obvious objection that the four rates are nearly collinear. It is
simply not a panel row.

Marker radius is proportional to a group's listening hours, so a group resting on 30 h
reads as twice one resting on 15 h. `p_FDR` is Benjamini-Hochberg within each row.

In [ ]:
# Three rows rather than four now the partial row is not drawn, and the height is kept
# near the four-row figure's -- so every panel gains, the top pair most.
MAIN_FIGSIZE = (6.27,7.5)
EXEMPLAR = "phoneme"          # which unit's stacked spectra fill the top-right panel

# Four panels across 6.27 in leaves ~1.3 in of axes each, so the scatter rows drop
# below the house body size and the tick locators are capped -- the default locator
# puts six labels on a 1.3 in axis.
SCATTER_LABEL_FONTSIZE = 7.5
SCATTER_TICK_FONTSIZE = 7
ROW_TITLE_FONTSIZE = 8

# Marker RADIUS is proportional to listening hours. matplotlib's `s` is the marker
# AREA in points squared, so passing (k * hours) ** 2 is what makes the radius, not the
# area, linear in hours.
MARKER_RADIUS_PER_HOUR = 0.28

# Axes are padded past the data on both axes so markers at the extremes are not sliced
# by the spines -- at these marker sizes the default 5% margin is not enough.
AXIS_MARGIN = 0.14

# Ticks fixed per unit column and per band row, rather than left to a locator. The
# locator picks its own round numbers from each panel's own range, so the four panels
# of a row ended up on four different ladders and could not be read against one
# another; these are shared down a column and across a row.
UNIT_XTICKS = {
    "iu": [0.2, 0.3, 0.4, 0.5, 0.6],
    "word": [2.0, 2.2, 2.4, 2.6, 2.8],
    "syllable": [2.0, 2.5, 3.0, 3.5, 4.0],
    "phoneme": [6, 8, 10, 12],
}
BAND_YTICKS = {
    "delta": [0.4, 0.5, 0.6, 0.7, 0.8],
    "theta": [6.5, 7.0, 7.5, 8.0, 8.5],
}


def tick_limits(values, ticks, margin=AXIS_MARGIN):
    """Limits covering both the data and every requested tick, with the usual pad.

    Either can be the binding one: the theta peaks run down to 6.05, below that row's
    lowest tick, while its highest tick at 8.5 sits well above the largest peak. Taking
    the union keeps every tick on the axis without ever clipping a marker.
    """
    low = min(float(np.min(values)), min(ticks))
    high = max(float(np.max(values)), max(ticks))
    pad = margin * (high - low)
    return low - pad, high + pad


def scatter_panel(ax, x, y, hours, r, p, *, xlabel=None, ylabel=None, headroom=0.0,
                  corner="upper left", xticks=None, yticks=None):
    """One rate-versus-measure panel."""
    ax.scatter(x, y, s=(MARKER_RADIUS_PER_HOUR * hours) ** 2, color=INK, alpha=0.55,
               linewidths=0, zorder=2)
    slope, intercept = np.polyfit(x, y, 1)
    span = np.linspace(x.min(), x.max(), 2)
    ax.plot(span, slope * span + intercept, color=MEDIAN_COLOUR, lw=1.1, zorder=3)
    # `headroom` lifts the top of the axes by a fraction of the data range so the
    # annotation has empty space to sit in. Rows whose points climb left to right leave
    # that corner free on their own and need none.
    ax.margins(x=AXIS_MARGIN, y=AXIS_MARGIN)
    if headroom and yticks is None:
        # An explicit limit overrides the y margin, so the padding is reapplied here.
        low, high = y.min(), y.max()
        ax.set_ylim(low - AXIS_MARGIN * (high - low), high + headroom * (high - low))
    x_anchor, align = (0.05, "left") if corner == "upper left" else (0.95, "right")
    # Three decimals would print a small p as 0.000, so below 0.001 it is given as a bound.
    p_text = "< 0.001" if p < 0.001 else f"= {p:.3f}"
    ax.text(x_anchor, 0.95, f"$r$ = {r:+.2f}\n$p_{{FDR}}$ {p_text}",
            transform=ax.transAxes, va="top", ha=align,
            fontsize=SCATTER_TICK_FONTSIZE, color=INK if p < 0.05 else MUTED)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=SCATTER_LABEL_FONTSIZE)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=SCATTER_LABEL_FONTSIZE)
    # Fixed ticks where they are given, a locator where they are not -- setting ticks
    # replaces the locator outright, so the two cannot both be applied.
    if xticks is None:
        ax.xaxis.set_major_locator(MaxNLocator(3))
    else:
        ax.set_xlim(*tick_limits(x, xticks))
        ax.set_xticks(xticks)
    if yticks is None:
        ax.yaxis.set_major_locator(MaxNLocator(4))
    else:
        ax.set_ylim(*tick_limits(y, yticks))
        ax.set_yticks(yticks)
    ax.tick_params(labelsize=SCATTER_TICK_FONTSIZE)
    bare(ax)


def fit_lift(frequencies_out, floor):
    """What to add to a drawn fit so it lands on the row's measured trace.

    The rows draw the coherence as measured, floor included. When the fit is of the
    excess it has to be lifted back onto that trace; when it is of the measured
    spectrum, as `FIT_ON_MEASURED` has it, it needs no lift.
    """
    if FIT_ON_MEASURED:
        return 0.0
    return np.interp(frequencies_out, union_frequencies, floor)


def draw_group_rows(ax, subset, colour, offset=1.05, captions=False):
    """One group per row: the measured spectrum, both band fits and their peaks, and
    the group's mean rate as a rule spanning its own row.

    Shared by the main figure's panel (b) and the supplementary figure, so a row is
    drawn identically in both and any difference between them is the grouping.
    """
    for height, (position, row) in enumerate(subset.iterrows()):
        measured = group_real_rms[position]
        floor = group_null_median[position]
        fit_values = measured if FIT_ON_MEASURED else measured - floor
        scale = GROUP_ROW_SCALE if COMMON_ROW_SCALE else measured.max()
        trace = measured / scale
        baseline = height * offset

        ax.fill_between(union_frequencies, baseline, baseline + trace,
                        color=HAIRLINE, alpha=0.45, lw=0, zorder=2)
        ax.plot(union_frequencies, baseline + trace, color=INK, lw=0.7, zorder=3)
        ax.axhline(baseline, color=HAIRLINE, lw=0.4, zorder=1)

        for band, marker in (("delta", "o"), ("theta", "s")):
            within = band_within_union[band]
            band_f = union_frequencies[within]
            coefficients, span = band_polynomial_fit(band_f, fit_values[within])
            dense = np.linspace(band_f[0], band_f[-1], 200)
            ax.plot(dense, baseline + (evaluate_band_fit(coefficients, span, dense)
                                       + fit_lift(dense, floor)) / scale,
                    color=MEDIAN_COLOUR, lw=1.0, alpha=0.9, zorder=4)
            peak = row[f"peak_{band}_bandfit"]
            ax.plot(peak, baseline + (evaluate_band_fit(coefficients, span, peak)
                                      + fit_lift(peak, floor)) / scale,
                    marker, color=MEDIAN_COLOUR, markersize=3.2,
                    markeredgecolor="white", markeredgewidth=0.5, zorder=5)

        ax.plot([row.mean_rate] * 2, [baseline, baseline + 1.0], color=colour, lw=1.4,
                solid_capstyle="butt", zorder=6)
        if captions:
            yield height, position, row, baseline
    ax.set_xlim(*SPECTRUM_RANGE)
    for low, high in GROUP_BANDS.values():
        ax.axvspan(low, high, color=HAIRLINE, alpha=0.3, lw=0, zorder=0)
    ax.set_yticks([])
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)


fig = plt.figure(figsize=MAIN_FIGSIZE)
# Row 1 carries the two large panels; rows 2-3 are the scatter rows. Four columns, so
# each scatter row gives one panel per unit.
#
# Both spacings are fractions of the AVERAGE cell, not of the cell they sit beside.
# With a top row nearly three times a scatter row, hspace=0.95 opened a gutter between
# the scatter rows taller than the rows themselves; half that still clears each row's
# x-label and the next row's title. Columns 2-4 carry no y-tick labels -- only the
# outermost x-tick label overhangs -- so the horizontal gutter can be tighter still,
# and the width it gives back goes to the panels.
outer = fig.add_gridspec(3, 4, height_ratios=[2.5, 1.15, 1.15], hspace=0.5,
                         wspace=0.28)

# --- (a) stimulus-level heatmap ---------------------------------------------------
ax_heat = fig.add_subplot(outer[0, 0:2])
row_order = np.argsort(unit_rates["Syllables"])
heat = stimulus_norm[:, in_band][row_order]
# Clipped to the bulk: the most extreme cells reach well past the 99th percentile and
# on an autoscale would take most of the colour range.
vmin, vmax = np.percentile(heat, [1, 99])
ax_heat.pcolormesh(frequencies[in_band], np.arange(len(heat)), heat, shading="auto",
                   cmap="cividis", rasterized=True, vmin=vmin, vmax=vmax)
ax_heat.set_xlim(*BAND)
ax_heat.set_yticks([])
ax_heat.set_xlabel("Frequency (Hz)", fontsize=SCATTER_LABEL_FONTSIZE)
ax_heat.set_ylabel("Stimulus\n(ordered by syllable rate)", fontsize=SCATTER_LABEL_FONTSIZE)
ax_heat.tick_params(labelsize=SCATTER_TICK_FONTSIZE)
ax_heat.set_title("RMS coherence for individual stimuli", fontsize=ROW_TITLE_FONTSIZE)
# One marker per stimulus per unit, at that stimulus's rate on the frequency axis --
# rates reordered with the rows, not after. A thin white edge holds the marker together
# where the plate behind it is pale.
for (label, rates), colour in zip(unit_rates.items(), UNIT_COLOURS):
    ax_heat.scatter(rates[row_order], np.arange(len(heat)), s=5, c=colour, marker="o",
                    linewidths=0.4, edgecolors="w", label=label, zorder=3)
ax_heat.legend(loc="upper right", fontsize=6, frameon=True, facecolor="white",
               framealpha=0.85, edgecolor="none", handletextpad=0.2,
               labelspacing=0.25, borderpad=0.35, markerscale=1.4)

# --- (b) stacked group spectra for the exemplar unit ------------------------------
ax_stack = fig.add_subplot(outer[0, 2:4])
exemplar_label = {slug: label for label, slug in UNIT_SLUGS.items()}[EXEMPLAR]
exemplar_colour = UNIT_COLOURS[list(UNIT_SLUGS).index(exemplar_label)]
subset = group_table[group_table.unit == EXEMPLAR].sort_values("mean_rate")
list(draw_group_rows(ax_stack, subset, exemplar_colour))

ax_stack.set_xticks(np.arange(0, SPECTRUM_RANGE[1] + 1, 10))
# No headroom: the limits stop just above the top trace so every row is drawn at the
# same height. The legend is allowed to overlap the top traces instead.
ax_stack.set_ylim(-0.08, (len(subset) - 1) * 1.05 + 1.12)
ax_stack.set_xlabel("Frequency (Hz)", fontsize=SCATTER_LABEL_FONTSIZE)
ax_stack.set_title(f"RMS coherence for\nstimuli grouped by "
                   f"{UNIT_LONG_NAMES[exemplar_label]} rate", fontsize=ROW_TITLE_FONTSIZE)
ax_stack.tick_params(axis="x", labelsize=SCATTER_TICK_FONTSIZE)
ax_stack.legend(handles=[
    Line2D([0], [0], color=INK, lw=0.8, label="Coherence spectrum"),
    Line2D([0], [0], color=MEDIAN_COLOUR, lw=1.0, marker="o", markersize=3.2,
           label="Fit + peak, 0.2-2 Hz"),
    Line2D([0], [0], color=MEDIAN_COLOUR, lw=1.0, marker="s", markersize=3.2,
           label="Fit + peak, 4-15 Hz"),
    Line2D([0], [0], color=exemplar_colour, lw=1.4,
           label=f"Avg. {UNIT_LONG_NAMES[exemplar_label]} rate"),
], loc="upper right", fontsize=5.5, frameon=True, facecolor="white", framealpha=0.9,
    edgecolor="none", handlelength=1.4, labelspacing=0.25, borderpad=0.35)

# --- (c)-(e) the scatter rows -----------------------------------------------------
# Two passes per row: the first collects every statistic so the family size is known,
# the second draws. A single pass cannot correct, because panel one would have to be
# annotated before panel four had been computed.
main_rows = []


def collect(measure, control_slug=None):
    """Every unit's statistic for one row, FDR-corrected within the row."""
    stats = []
    for label, slug in UNIT_SLUGS.items():
        if control_slug is not None and slug == control_slug:
            continue                     # a rate cannot be partialled out of itself
        groups = group_table[group_table.unit == slug].sort_values("mean_rate")
        x = groups.mean_rate.to_numpy()
        y = groups[measure].to_numpy()
        if control_slug is None:
            result = pearsonr(x, y)
            r, pvalue = result.statistic, result.pvalue
        else:
            covariate = groups[f"mean_rate_{control_slug}"].to_numpy()
            r, pvalue = partial_correlation(x, y, covariate)
            x, y = control_residuals(x, covariate), control_residuals(y, covariate)
            # The residual scatter and the reported partial r are the same number.
            assert abs(pearsonr(x, y).statistic - r) < 1e-9, slug
        stats.append(dict(label=label, slug=slug, x=x, y=y, n=len(x),
                          hours=groups.hours.to_numpy(), r=r, p=pvalue))
    for entry, corrected in zip(stats, correct([e["p"] for e in stats])):
        entry["p_fdr"] = corrected
    return stats


def draw_row(stats, gridspec_row, first_column, row_name, ylabel, band=None,
             **panel_kwargs):
    axes_row = []
    for offset, entry in enumerate(stats):
        column = first_column + offset
        ax = fig.add_subplot(outer[gridspec_row, column],
                             sharey=axes_row[0] if axes_row else None)
        axes_row.append(ax)
        main_rows.append({"row": row_name, "unit": entry["slug"], "n": entry["n"],
                          "r": entry["r"], "p": entry["p"], "p_fdr": entry["p_fdr"]})
        suffix = " (resid.)" if "resid" in ylabel else " (Hz)"
        scatter_panel(ax, entry["x"], entry["y"], entry["hours"], entry["r"],
                      entry["p_fdr"], xlabel=f"{entry['label']} rate{suffix}",
                      ylabel=ylabel if offset == 0 else None,
                      xticks=UNIT_XTICKS[entry["slug"]] if band else None,
                      yticks=BAND_YTICKS[band] if band else None, **panel_kwargs)
        if offset:
            ax.tick_params(labelleft=False)
    return axes_row


delta_axes = draw_row(collect("peak_delta_bandfit"), 1, 0, "delta", "Peak 0.2-2 Hz",
                      band="delta")
theta_axes = draw_row(collect("peak_theta_bandfit"), 2, 0, "theta", "Peak 4-15 Hz",
                      band="theta", corner="upper right")

# Computed, reported, not drawn. The delta effect surviving with intonation-unit rate
# partialled out is the answer to the collinearity objection, so it stays in the
# printed table even though it is no longer a row of panels.
for entry in collect("peak_delta_bandfit", control_slug="iu"):
    main_rows.append({"row": "delta | IU (not drawn)", "unit": entry["slug"],
                      "n": entry["n"], "r": entry["r"], "p": entry["p"],
                      "p_fdr": entry["p_fdr"]})

# Row titles and the c/d/e labels are placed in FIGURE coordinates, not on their axes:
# row (d) starts one column in, so an axes-relative label would sit a whole column
# right of (c)'s. Taking x from the leftmost column puts all three in one line.
ROW_TITLES = [
    (delta_axes, "c)",
     "Correlation between speech unit rates and low-delta peak coherence frequency"),
    (theta_axes, "d)",
     "Correlation between speech unit rates and theta-alpha peak coherence frequency"),
]
label_x = delta_axes[0].get_position().x0 - 0.055
for axes_row, label, title in ROW_TITLES:
    top = max(ax.get_position().y1 for ax in axes_row)
    # Centred over the panels the row actually occupies, so (d)'s title sits over its
    # three rather than over the full width.
    left, right = axes_row[0].get_position().x0, axes_row[-1].get_position().x1
    fig.text((left + right) / 2, top + 0.010, title, ha="center", va="bottom",
             fontsize=ROW_TITLE_FONTSIZE)
    fig.text(label_x, top + 0.010, label, ha="left", va="bottom",
             fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold")

# Per-panel x offsets: (a) has to clear its two-line y-label, (b) carries neither
# ticks nor label on the left, so the same offset there would put the marker under its
# own title -- which is nearly as wide as the panel.
# y clears the panel titles: at this panel height the default title pad puts a title's
# top near 1.07 in axes coordinates, and the marker is anchored by its bottom -- so
# 1.05 sat inside the title's own band, which (b)'s nearly-full-width title showed.
for ax, label, offset in [(ax_heat, "a)", -0.18), (ax_stack, "b)", -0.06)]:
    ax.text(offset, 1.08, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = f"speech_rate_and_coherence_{REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

main_figure_correlations = pd.DataFrame(main_rows)
print(main_figure_correlations.pivot(index="row", columns="unit",
                                     values=["r", "p", "p_fdr"]).round(3).to_string())
print("\np_FDR: Benjamini-Hochberg within each row -- family of 4 in (c) and (d), 3 in "
      "the\npartial row, where IU cannot be partialled out of itself. The panels "
      "annotate the\ncorrected value; `p` above is uncorrected. The partial row is "
      "not drawn.")

## Figure 2 -- grouped spectra, all four units

Panel (b) of the main figure, for every unit rather than the exemplar, as a 2x2 square
so the four groupings can be read against each other. Every row is drawn by the same
`draw_group_rows` the main figure uses, so a difference between a panel here and the
main figure's (b) is the grouping and nothing else.

Unlike the main panel the rows **are** captioned here: this is the figure a reader comes
to for the detail, and a group's mean rate and listening hours are what the main
figure's scatter markers encode only as a radius. The captions live inside the panel, in
the 15-30 Hz stretch, which is flat for every group -- so they cost no data and free the
y axis entirely, which also stops the right-hand panels' labels colliding with the
left-hand panels' traces.

In [ ]:
SUPP_FIGSIZE = (6.27, 7.4)
SUPP_OFFSET = 1.05

# 5 pt is not a house size -- it is what fits two lines of caption into the 15 Hz of
# panel available at this width without running past the axis.
CAPTION_X = 15.6
CAPTION_FONTSIZE = 5

# One y-limit for every panel, set by the largest grouping. Without it a trace would be
# drawn shorter in one panel than another purely because that unit happened to split
# into one more group -- which is exactly the across-panel comparison this figure is for.
max_groups = max(int((group_table.unit == slug).sum()) for slug in UNIT_SLUGS.values())
supp_ylim = (-0.08, (max_groups - 1) * SUPP_OFFSET + 1.12)

fig, axes = plt.subplots(2, 2, figsize=SUPP_FIGSIZE, sharex=True)
axes = axes.ravel()

for ax, (label, slug) in zip(axes, UNIT_SLUGS.items()):
    subset = group_table[group_table.unit == slug].sort_values("mean_rate")
    colour = UNIT_COLOURS[list(UNIT_SLUGS).index(label)]

    for _, _, row, baseline in draw_group_rows(ax, subset, colour, offset=SUPP_OFFSET,
                                               captions=True):
        ax.text(CAPTION_X, baseline + 0.98,
                f"Listening hours: {row.hours:.0f} "
                f"(num. stimuli: {int(row.n_stimuli)})\n"
                f"Average {UNIT_LONG_NAMES[label]} rate: {row.mean_rate:.2f} Hz",
                fontsize=CAPTION_FONTSIZE, va="top", ha="left", color=INK,
                linespacing=1.35, zorder=7)

    ax.set_ylim(*supp_ylim)
    ax.set_xticks(np.arange(0, SPECTRUM_RANGE[1] + 1, 5))
    ax.tick_params(axis="x", labelsize=SCATTER_TICK_FONTSIZE)
    ax.set_title(f"Stimuli grouped by {UNIT_LONG_NAMES[label]} rate",
                 fontsize=ROW_TITLE_FONTSIZE)
    ax.set_ylabel("RMS coherence", fontsize=SCATTER_LABEL_FONTSIZE)

for ax in axes[2:]:
    ax.set_xlabel("Frequency (Hz)", fontsize=SCATTER_LABEL_FONTSIZE)

# Panel labels sit closer to the spine than the main figure's -0.18: this figure's y
# axis carries no ticks, so there is nothing between the label and the plate.
for ax, label in zip(axes, ("a)", "b)", "c)", "d)")):
    ax.text(-0.07, 1.02, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

# One legend for the whole figure rather than four. The three shared encodings fill the
# first column and the four rate colours the rest, so a reader finds the coloured rule
# for the panel they are looking at without four boxes competing with the traces.
handles = [
    Line2D([0], [0], color=INK, lw=0.8, label="Coherence spectrum"),
    Line2D([0], [0], color=MEDIAN_COLOUR, lw=1.0, marker="o", markersize=3.2,
           label="Fit + peak, 0.2-2 Hz"),
    Line2D([0], [0], color=MEDIAN_COLOUR, lw=1.0, marker="s", markersize=3.2,
           label="Fit + peak, 4-15 Hz"),
] + [Line2D([0], [0], color=UNIT_COLOURS[i], lw=1.4,
            label=f"Avg. {UNIT_LONG_NAMES[label]} rate")
     for i, label in enumerate(UNIT_SLUGS)]

fig.tight_layout(rect=[0, 0, 1, 0.965])
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.012), ncol=3,
           fontsize=6, frameon=False, handlelength=1.6, labelspacing=0.35,
           columnspacing=1.8)

stem = f"grouped_spectra_by_unit_{REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

## Figure 3 -- how strong the band is, not where it peaks

The main figure's rows (c)-(e) ask *where* in a band the response sits. This one asks
*how strong* it is: each group's RMS coherence averaged over the band's bins, against
the same four unit rates, with the same family-of-four BH correction within each row.

The two questions come apart, which is the point of drawing it against rows (c) and (e).

A magnitude, unlike a peak frequency, is displaced directly by the noise floor beneath
it, and that floor tracks listening time, which is not uniform across groups. So the
cell runs every test a second time on the null-corrected magnitude and prints both: if
the floor were manufacturing a trend, the two would disagree.

Written as a function because the exploratory notebook draws the same figure again with
listening time and with the floor partialled out. Those variants are not reproduced
here, but the covariate hook is kept so they can be.

In [ ]:
RMS_FIGSIZE = (6.27, 3.9)
RMS_BAND_ROWS = [("delta", "0.2-2 Hz", "upper right"), ("theta", "4-15 Hz", "upper left")]


def draw_band_rms_figure(stem, control=None, control_label=None, measure="rms",
                         name="RMS", axis_label="RMS coh.", title_measure="RMS coherence"):
    """A band measure against each unit rate, two rows of four -- band RMS by default.

    `measure` names the group_table columns drawn, ``{measure}_{band}`` and its
    null-corrected ``{measure}_{band}_excess``; the three strings label the rows, the
    axes and the titles, so another strength measure draws through the same code.

    `control` is a function (band, groups) -> covariate array. When given, BOTH axes
    carry residuals after that covariate is regressed out, and the annotation becomes
    the partial Pearson r -- the same construction as the main figure's row (d), so the
    residual scatter is its exact counterpart.
    """
    fig, axes = plt.subplots(2, 4, figsize=RMS_FIGSIZE)
    rows = []

    for row_index, (band, band_label, corner) in enumerate(RMS_BAND_ROWS):
        stats = []
        for label, slug in UNIT_SLUGS.items():
            groups = group_table[group_table.unit == slug].sort_values("mean_rate")
            x = groups.mean_rate.to_numpy()
            y = groups[f"{measure}_{band}"].to_numpy()
            excess_y = groups[f"{measure}_{band}_excess"].to_numpy()
            if control is None:
                result = pearsonr(x, y)
                r, pvalue = result.statistic, result.pvalue
                r_excess = pearsonr(x, excess_y).statistic
            else:
                covariate = control(band, groups)
                r, pvalue = partial_correlation(x, y, covariate)
                r_excess = partial_correlation(x, excess_y, covariate)[0]
                x, y = control_residuals(x, covariate), control_residuals(y, covariate)
                assert abs(pearsonr(x, y).statistic - r) < 1e-9, (band, slug)
            stats.append(dict(label=label, slug=slug, x=x, y=y, n=len(x),
                              hours=groups.hours.to_numpy(), r=r, p=pvalue,
                              r_excess=r_excess))
        for entry, corrected in zip(stats, correct([e["p"] for e in stats])):
            entry["p_fdr"] = corrected

        for column, entry in enumerate(stats):
            ax = axes[row_index, column]
            if column:
                ax.sharey(axes[row_index, 0])
            rows.append({"row": f"{band} {name}", "unit": entry["slug"], "n": entry["n"],
                         "r": entry["r"], "p": entry["p"], "p_fdr": entry["p_fdr"],
                         "r_excess": entry["r_excess"]})
            suffix = " (resid.)" if control is not None else " (Hz)"
            scatter_panel(ax, entry["x"], entry["y"], entry["hours"], entry["r"],
                          entry["p_fdr"], xlabel=f"{entry['label']} rate{suffix}",
                          ylabel=(f"{axis_label}\n{band_label}"
                                  + ("\n(resid.)" if control is not None else ""))
                          if column == 0 else None,
                          headroom=0.75, corner=corner)
            if column:
                ax.tick_params(labelleft=False)

    # Row titles and panel labels in FIGURE coordinates, as in the main figure, so both
    # rows' labels line up with each other rather than with their own axes.
    fig.tight_layout(rect=[0, 0, 1, 0.93], h_pad=3.2)
    tail = f", {control_label} partialled out" if control_label else ""
    titles = [(axes[0], "a)", f"Correlation between speech unit rates and low-delta "
                              f"{title_measure}{tail}"),
              (axes[1], "b)", f"Correlation between speech unit rates and theta-alpha "
                              f"{title_measure}{tail}")]
    label_x = axes[0, 0].get_position().x0 - 0.055
    for axes_row, label, title in titles:
        top = max(ax.get_position().y1 for ax in axes_row)
        left, right = axes_row[0].get_position().x0, axes_row[-1].get_position().x1
        fig.text((left + right) / 2, top + 0.012, title, ha="center", va="bottom",
                 fontsize=ROW_TITLE_FONTSIZE)
        fig.text(label_x, top + 0.012, label, ha="left", va="bottom",
                 fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold")

    fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
    fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
    plt.show()

    table = pd.DataFrame(rows)
    print(table.pivot(index="row", columns="unit",
                      values=["r", "p", "p_fdr"]).round(3).to_string())
    print(f"\nDrawn on the measured band {name}; the same tests on the null-corrected excess:")
    print(table.pivot(index="row", columns="unit",
                      values=["r", "r_excess"]).round(2).to_string())
    print(f"largest disagreement {np.abs(table.r - table.r_excess).max():.2f} in r.")
    return table


band_rms_correlations = draw_band_rms_figure(f"band_rms_vs_rate_{REFERENCE}")

## Figure 4 -- the height of the band's peak

Figure 3 measures a band's strength as its RMS coherence averaged over every bin. This
one takes the coherence at the band's peak instead: the height of the same order-4
polynomial fit the main figure uses to locate the peak, read off at that peak. So it is
the y-coordinate of the point whose x-coordinate the main figure's rows (c)-(e)
correlate with rate.

The two strength measures can differ. A band can rise at its peak while its average
stays put if the peak sharpens, and vice versa if it broadens. The cell prints how
closely they agree across groups before the figure.

As with the RMS, the measure is taken on the measured spectrum (`FIT_ON_MEASURED`) and
the tests are repeated on the null-corrected excess: the excess fit is evaluated at the
same peak frequency, so the two versions describe the same point of the spectrum.

In [ ]:
# The height of each group's band fit at its own peak: the fit's value where the main
# figure's peak frequency sits. The excess version evaluates the fit of the
# null-corrected spectrum at that same frequency, so the two describe one point.
for band, within in band_within_union.items():
    band_f = union_frequencies[within]
    heights, excess_heights = [], []
    for measured, excess, peak in zip(band_source[:, within], group_excess[:, within],
                                      group_table[f"peak_{band}_bandfit"]):
        at_peak = np.array([peak])
        heights.append(float(evaluate_band_fit(*band_polynomial_fit(band_f, measured),
                                               at_peak)[0]))
        excess_heights.append(float(evaluate_band_fit(*band_polynomial_fit(band_f, excess),
                                                      at_peak)[0]))
    group_table[f"peakcoh_{band}"] = heights
    group_table[f"peakcoh_{band}_excess"] = excess_heights

# How far the two strength measures agree, group by group, within each unit's partition.
print("Pearson r between peak height and band RMS across the groups of each unit:")
for band in GROUP_BANDS:
    agreement = {slug: pearsonr(group_table.loc[group_table.unit == slug, f"peakcoh_{band}"],
                                group_table.loc[group_table.unit == slug, f"rms_{band}"]).statistic
                 for slug in UNIT_SLUGS.values()}
    print(f"  {band:6s} " + "  ".join(f"{slug} {r:+.2f}" for slug, r in agreement.items()))
print()

band_peak_height_correlations = draw_band_rms_figure(
    f"band_peak_height_vs_rate_{REFERENCE}", measure="peakcoh", name="peak height",
    axis_label="Peak coh.", title_measure="peak coherence")